# 03 Line Following With Default YOLO Detection

This notebook launches the combined line-following and object-detection panel from `scripts/teleop_server_article_contour_panel_with_yolo.py`.

It reuses:

- `.jetcar_motor_calibration.json` from notebook `01`
- `.contour_follow_settings.json` for the contour detection sliders

This version keeps the default YOLO model `yolo11n.pt` for general object detection, while auto driving still follows the contour detector on the lower ROI.


## Safety

Start on a stand, then move to the track only after camera, serial, YOLO, and test preset buttons all look correct. Keep `Stop Auto` or `Stop Motors` ready.

This notebook is for the default YOLO demo. Use notebook `04` when you want the custom traffic-sign stop action.


In [15]:
import glob
import os
import socket
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
video_devices = sorted(glob.glob('/dev/video*'))
serial_candidates = [p for p in ('/dev/ttyUSB0', '/dev/ttyTHS1', '/dev/ttyTHS2') if os.path.exists(p)]
hostname = socket.gethostname()
ip_addresses = sorted({addr[-1][0] for addr in socket.getaddrinfo(hostname, None, family=socket.AF_INET)})
yolo_model_candidates = sorted(str(path.relative_to(project_root)) for path in project_root.rglob('*.pt'))

print('Project root:', project_root)
print('Video devices:', video_devices or ['none found'])
print('Serial candidates:', serial_candidates or ['none found'])
print('Calibration file exists:', (project_root / '.jetcar_motor_calibration.json').exists())
print('Contour settings file exists:', (project_root / '.contour_follow_settings.json').exists())
print('YOLO model files:', yolo_model_candidates or ['none found in this repo'])
print('Host IPv4:', ip_addresses or ['127.0.0.1'])


Project root: /home/orin/JetCar
Video devices: ['/dev/video0', '/dev/video1', '/dev/video2']
Serial candidates: ['/dev/ttyUSB0', '/dev/ttyTHS1', '/dev/ttyTHS2']
Calibration file exists: False
Contour settings file exists: False
YOLO model files: ['yolo11n.pt']
Host IPv4: ['127.0.1.1']


In [16]:
camera_source = 'auto'
host = '0.0.0.0'
http_port = 8765
serial_port = 'auto'
baudrate = 115200
sensor_id = 0
device_index = 0
frame_width = 1280
frame_height = 720
warmup_frames = 12

yolo_model = 'yolo11n.pt'
yolo_conf = 0.35
yolo_imgsz = 640
yolo_every_n = 1
yolo_device = ''
yolo_max_det = 20

print('Leave camera_source and serial_port as \'auto\' to detect working hardware, or override them manually if needed.')
print('Current default YOLO model:', yolo_model)


Leave camera_source and serial_port as 'auto' to detect working hardware, or override them manually if needed.
Current default YOLO model: yolo11n.pt


In [17]:
import shlex
import subprocess
import sys
import time
from IPython.display import HTML, display

if '_article_contour_yolo_process' not in globals():
    _article_contour_yolo_process = None

panel_log_path = project_root / '.teleop_server_article_contour_panel_with_yolo.notebook.log'

def panel_python() -> str:
    candidate = project_root / '.venv' / 'bin' / 'python'
    return str(candidate) if candidate.exists() else sys.executable


def panel_command() -> list[str]:
    return [
        panel_python(),
        str(project_root / 'scripts' / 'teleop_server_article_contour_panel_with_yolo.py'),
        '--host', host,
        '--http-port', str(http_port),
        '--camera-source', camera_source,
        '--sensor-id', str(sensor_id),
        '--device-index', str(device_index),
        '--width', str(frame_width),
        '--height', str(frame_height),
        '--warmup-frames', str(warmup_frames),
        '--port', serial_port,
        '--baudrate', str(baudrate),
        '--yolo-model', str(yolo_model),
        '--yolo-conf', str(yolo_conf),
        '--yolo-imgsz', str(yolo_imgsz),
        '--yolo-every-n', str(yolo_every_n),
        '--yolo-device', str(yolo_device),
        '--yolo-max-det', str(yolo_max_det),
    ]


def stop_article_contour_yolo_panel() -> None:
    global _article_contour_yolo_process
    if _article_contour_yolo_process is None:
        return
    if _article_contour_yolo_process.poll() is None:
        _article_contour_yolo_process.terminate()
        try:
            _article_contour_yolo_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            _article_contour_yolo_process.kill()
            _article_contour_yolo_process.wait(timeout=3)
    _article_contour_yolo_process = None


def browser_url() -> str:
    browser_host = '127.0.0.1' if host == '0.0.0.0' else host
    return f'http://{browser_host}:{http_port}'


print('Equivalent terminal command:')
print(shlex.join(panel_command()))


Equivalent terminal command:
/home/orin/JetCar/.venv-usb/bin/python /home/orin/JetCar/scripts/teleop_server_article_contour_panel_with_yolo.py --host 0.0.0.0 --http-port 8765 --camera-source auto --sensor-id 0 --device-index 0 --width 1280 --height 720 --warmup-frames 12 --port auto --baudrate 115200 --yolo-model yolo11n.pt --yolo-conf 0.35 --yolo-imgsz 640 --yolo-every-n 1 --yolo-device '' --yolo-max-det 20


In [18]:
stop_article_contour_yolo_panel()
cmd = panel_command()
print('Starting line following + default YOLO panel...')
print(shlex.join(cmd))
with open(panel_log_path, 'w') as log_file:
    _article_contour_yolo_process = subprocess.Popen(
        cmd,
        cwd=project_root,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
time.sleep(2.0)
if _article_contour_yolo_process.poll() is not None:
    raise RuntimeError(f'line following + default YOLO panel exited early; check {panel_log_path}')
print('Line following + default YOLO panel running at:', browser_url())
print('YOLO model:', yolo_model)
if ip_addresses:
    print('LAN example:', f'http://{ip_addresses[0]}:{http_port}')
display(HTML(
    f'<p><a href="{browser_url()}" target="_blank">Open the line following + default YOLO panel in a new tab</a></p>'
    f'<iframe src="{browser_url()}" width="100%" height="900" style="border:1px solid #ccc; border-radius:12px;"></iframe>'
))


Starting line following + default YOLO panel...
/home/orin/JetCar/.venv-usb/bin/python /home/orin/JetCar/scripts/teleop_server_article_contour_panel_with_yolo.py --host 0.0.0.0 --http-port 8765 --camera-source auto --sensor-id 0 --device-index 0 --width 1280 --height 720 --warmup-frames 12 --port auto --baudrate 115200 --yolo-model yolo11n.pt --yolo-conf 0.35 --yolo-imgsz 640 --yolo-every-n 1 --yolo-device '' --yolo-max-det 20
Line following + default YOLO panel running at: http://127.0.0.1:8765
YOLO model: yolo11n.pt
LAN example: http://127.0.1.1:8765


In [19]:
print('Last line following + default YOLO panel log lines:')
if panel_log_path.exists():
    print('\n'.join(panel_log_path.read_text().splitlines()[-60:]))
else:
    print('No log file yet.')


Last line following + default YOLO panel log lines:



In [20]:
# stop_article_contour_yolo_panel()
# print('Line following + default YOLO panel stopped.')


## What To Do In The Panel

1. make sure `Camera`, `Serial`, and `YOLO` become ready
2. confirm the RGB image is live and default YOLO boxes appear on detected objects
3. confirm the grayscale ROI, mask, and contour overlay still track the line well
4. use the preset test buttons to confirm the saved calibration still feels correct
5. adjust contour sliders if the line-follow view looks unstable
6. place the rover on the line and press `Auto Start`
7. watch both `Detection Metrics` and the RGB view while it drives
8. press `Stop Auto` or `Stop Motors` immediately if behavior looks wrong
9. move to `04_line_following_with_stop_sign_action.ipynb` when you want the custom stop-sign action

The contour detector still controls steering. Default YOLO adds general object detection and boxed labels on the RGB view.
